# Notebook 3: State-of-the-Art Transformers (BERT)
## Project: Multi-Paradigm AI vs. Human Text Detection

### Objective
To deploy a state-of-the-art (SOTA) Transformer architecture. While LSTMs read text left-to-right, BERT (Bidirectional Encoder Representations from Transformers) reads the entire sentence at once, allowing it to capture deep, bidirectional contextual relationships.



### The Transformer Advantage
We utilize the `bert-base-uncased` model. With 110 million parameters, it comes pre-trained on the entire English Wikipedia and BookCorpus. Our goal is to fine-tune this massive "understanding" of human language specifically for the binary classification task of AI detection.

In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import re
import torch
from torch.utils.data import Dataset,DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer,BertForSequenceClassification
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Maintaining reusability
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

In [3]:
# Loading data
df=pd.read_csv("/kaggle/input/datasets/matejkore/ai-detector-dataset/ai_detector_dataset.csv")
df.head()

,text,label
0,"Got take-out. Very friendly staff, reasonable ...",Human
1,Love this bar! Fun crowd and the staff are all...,Human
2,"After watching Whale Wars: Viking Shores, I ca...",Human
3,Kelly really wanted the new iPhone. She begged...,Human
4,Though this is the easiest way to secure a spo...,Human


In [4]:
# Replacing author names with binary labels
df['label']=[1 if author=='AI' else 0 for author in df['label']]
df.head()

,text,label
0,"Got take-out. Very friendly staff, reasonable ...",0
1,Love this bar! Fun crowd and the staff are all...,0
2,"After watching Whale Wars: Viking Shores, I ca...",0
3,Kelly really wanted the new iPhone. She begged...,0
4,Though this is the easiest way to secure a spo...,0


In [5]:
# Cleaning the texts
def text_cleaning(text):
    text=text.lower()
    text=re.sub(r'\n',' ',text)
    text=re.sub(r'\s+',' ',text)
    return text
df['text']=df['text'].apply(text_cleaning)
df.head()

,text,label
0,"got take-out. very friendly staff, reasonable ...",0
1,love this bar! fun crowd and the staff are all...,0
2,"after watching whale wars: viking shores, i ca...",0
3,kelly really wanted the new iphone. she begged...,0
4,though this is the easiest way to secure a spo...,0


In [6]:
# Dropping duplicates
df=df.drop_duplicates(subset='text')
df.duplicated(subset='text').sum()

np.int64(0)

In [7]:
# Splitting the dataset into train and text split
X_train,X_test,y_train,y_test=train_test_split(df['text'],df['label'],test_size=0.2,random_state=42,stratify=df['label'])
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((265651,), (265651,), (66413,), (66413,))

In [8]:
# Tokenizing the dataset
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings=tokenizer(X_train.tolist(),
                            padding=True,
                            truncation=True,
                            max_length=200)
test_encodings=tokenizer(X_test.tolist(),
                         padding=True,
                         truncation=True,
                         max_length=200)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Creating Dataset Object
class BertDataset(Dataset):
    def __init__(self,encodings,labels):
        self.encodings=encodings
        self.labels=labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self,idx):
        item={'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
              'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
              'label': torch.tensor(self.labels.iloc[idx], dtype=torch.long)}
        return item
train_dataset=BertDataset(train_encodings,y_train)
test_dataset=BertDataset(test_encodings,y_test)

In [10]:
# Creating DataLoader Object
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=16)

In [11]:
# Defining the BERT Model
model=BertForSequenceClassification.from_pretrained('bert-base-uncased',
                                                    num_labels=2)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer=AdamW(model.parameters(),lr=5e-5)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# Defining the training function
def train_model(model,loader):
    model.train()
    total_loss=0
    loop=tqdm(loader,desc="Training",leave=True)
    for batch in loop:
        input_ids=batch['input_ids'].to(device)
        attention_mask=batch['attention_mask'].to(device)
        labels=batch['label'].to(device)
        optimizer.zero_grad()
        outputs=model(input_ids=input_ids,
                      attention_mask=attention_mask,
                      labels=labels)
        loss=outputs.loss
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss/len(loader)

In [13]:
# Defining the evaluation function
def evaluate(model, loader):
    model.eval()
    correct=0
    total=0
    loop=tqdm(loader,desc="Evaluating",leave=True)
    with torch.no_grad():
        for batch in loop:
            input_ids=batch['input_ids'].to(device)
            attention_mask=batch['attention_mask'].to(device)
            labels=batch['label'].to(device)
            outputs=model(input_ids=input_ids,attention_mask=attention_mask)
            logits=outputs.logits
            preds=torch.argmax(logits,dim=1)
            correct+=(preds==labels).sum().item()
            total+=labels.size(0)
            loop.set_postfix(acc=correct/total)
    return correct/total

In [14]:
# Running the training loop
for epoch in range(1):
    loss=train_model(model,train_loader)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

Training: 100%|██████████| 16604/16604 [2:31:38<00:00,  1.83it/s, loss=0.138]   

Epoch 1, Loss: 0.2027


In [15]:
# Evaluating BERT performance on test data
accuracy=evaluate(model,test_loader)
print("BERT Accuracy:", accuracy)

Evaluating: 100%|██████████| 4151/4151 [11:51<00:00,  5.83it/s, acc=0.945]

BERT Accuracy: 0.9445138752954993


In [16]:
# Saving the model
save_directory="./saved_model"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Sucessfully saved to {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sucessfully saved to ./saved_model
